# MedMCQA + MedQA: deduplicated splits with three MedMCQA evaluation subsets (v6)

One call returns fixed, leak-free splits in one common format:

```python
from src.data.medqa_datasets import load_splits, build_messages
splits = load_splits("medmcqa")   # {"train", "hp_dev", "calib", "thresh", "test"}
medqa = load_splits("medqa")      # {"train", "val", "test"}
messages = build_messages(splits["train"][0])
```

**How to use this notebook:** run the cells top to bottom. Cell 1 is the only place that decides where data lives.
The notebook contains no pipeline code: cell 1 clones the team repo and imports it from `src/data/medqa_datasets/`.
It exists for analysis and for running on Colab; elsewhere, use the `.py` code directly. The data itself goes to
`$FME_SHARED_DIR/data` (Google Drive), never into git.

**Three MedMCQA evaluation subsets (v4), made test-like by a similarity filter (v5, threshold 0.85 in v6)**

The MedMCQA `val` split is replaced by three disjoint subsets carved from train, so that tuning, calibration and thresholding never
share questions and none of them touches `test` (the official MedMCQA validation set, used only for final numbers).

| Subset | Nominal size | Used for | Never used for |
|---|---|---|---|
| `hp_dev` | 2000 | choosing learning rate and LoRA rank | fitting T, choosing the threshold, final numbers |
| `calib` | 1500 | fitting the temperature T of each checkpoint | tuning, choosing the threshold, final numbers |
| `thresh` | 2500 | choosing the abstention threshold of each checkpoint | tuning, fitting T, final numbers |
| `test` | official validation set | final reported numbers only | any fitting or selection |

MedQA keeps its single `val` split (1000 rows). Subsets are carved with a fixed seed after sorting ids, so the result does not depend
on load order; a few rows per subset can be dropped by the leak guard, so actual sizes are slightly below nominal (see `stats.json`).

**Why the filter exists.** MedMCQA's authors built its dev/test sets so that questions with a Levenshtein similarity above 0.9 to
other questions were left out. The v4 diagnostic on the real data showed the consequence: 21.8% of random train rows have a different
train question at similarity 0.9 or more, against 0.5% of `test` rows (at 0.8: 34.6% vs 7.4%; median nearest similarity 0.71 vs 0.56).
Random subsets of train would therefore be easier and look more familiar to a trained model than `test`, which biases the fitted
temperature and abstention threshold. A candidate row is only eligible for `hp_dev`, `calib` or `thresh` if no other train
question has similarity at or above `SIMILARITY_FILTER["medmcqa"]["threshold"]` to it (identical text is ignored, the leak guard handles that). Because the comparison is made
against the whole train set before carving, the three subsets are also free of near-copies of each other.

The v5 run used 0.9 and removed the 0.9 gap completely (0.0% vs 0.6% for `test`), but at 0.8 the subsets were still easier than `test`
(hp_dev 20.0%, calib 14.9%, thresh 16.2% vs 8.0%). v6 uses 0.85 as a compromise: it closes more of that gap without pushing the subsets
below `test` the other way. The post-build table shows what the real shares are.

**Two-tier deduplication (from v3)**

| Where | Rule | Why |
|---|---|---|
| Inside train (and across the two train sets) | `exact_key`: same question **and** same options | Removes only true copies. Measured on the reference run: v1's question-only rule removed 30652 MedMCQA train rows, the exact rule removes 18106. The rest were distinct questions. |
| Train vs val/test, and inside val/test, and between the two datasets' eval sets | `leak_key`: long stems (10+ words) match on the question alone; short stems also need the same options | Strict where it matters: a reworded copy of an eval question must not survive in train. |

Earlier fixes kept from v2: pinned dataset commits (recorded in `stats.json`), a config guard in `load_splits`, and tests that check every row.

**Shared settings (all live in `src/data/medqa_datasets/data.py`).** Changing any of them changes the splits, so rebuild and re-share `splits/`.

| Setting | Meaning |
|---|---|
| `FME_SHARED_DIR` (env var) | Shared root. Data, ids and the HF cache are stored in `DATA_DIR` = `$FME_SHARED_DIR/data` (see `src/utils/paths.py`). Default `/shared`. |
| `SIMILARITY_FILTER` | Per dataset: similarity threshold and candidate-pool factor of the filter above (MedMCQA only). |
| `SEED`, `SPLIT_SIZES` | Shuffle seed and the named subsets carved from each train set (MedMCQA: `hp_dev`, `calib`, `thresh`; MedQA: `val`). |
| `SHORT_STEM_WORDS` | `leak_key` threshold: stems shorter than this also need identical options to match. |
| `DEDUPE_SCHEME` | Label of the dedupe rules (two-tier). Bump it if you change how keys are built. |
| `HF_REPOS`, `HF_REVISIONS` | Which datasets, and which commit of each (pinned below). |
| `SPLIT_PRIORITY`, `DATASET_ORDER` | Which copy survives a duplicate (test > val > hp_dev > calib > thresh > train; MedQA before MedMCQA). |

## Cell 1: setup

Clones the team repo (branch `BRANCH`) and puts it on `sys.path`, installs the pinned `datasets` and `rapidfuzz`,
mounts Drive (only when running on Colab) and sets `FME_SHARED_DIR`; the data goes to `DATA_DIR` = `$FME_SHARED_DIR/data`.
`FME_SHARED_DIR` must be set **before** `src.data.medqa_datasets` is imported, because `src/utils/paths.py` reads it once at import time.
Every teammate should point it at the same shared folder.


In [ ]:
!pip -q install datasets==5.2.0 rapidfuzz==3.14.6
import os, subprocess, sys

BRANCH = "main"   # <- the repo branch to run
REPO_URL = "https://github.com/tlelievre26/found-model-eng-team-4.git"
REPO_DIR = "/content/found-model-eng-team-4"
FME_SHARED_DIR = "/content/drive/MyDrive/shared"   # <- the one line to edit per team / machine
DATA_DIR = os.path.join(FME_SHARED_DIR, "data")    # where src.utils.paths puts the data

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "-q", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "-q", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-q", "FETCH_HEAD"], check=True)
sys.path.insert(0, REPO_DIR)

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass  # not on Colab: FME_SHARED_DIR just has to be a writable folder

os.environ["FME_SHARED_DIR"] = FME_SHARED_DIR
os.makedirs(DATA_DIR, exist_ok=True)
print("DATA_DIR =", DATA_DIR)


## Cell: build the splits

`build_all()` downloads both datasets (about 90 MB, cached under `DATA_DIR/hf_cache`), builds the splits and prints the
statistics. What to expect (same pinned commits as v3):

- MedMCQA: `hp_dev`, `calib` and `thresh` close to 2000 / 1500 / 2500 (a handful of rows lost to the leak guard), `test` 4168, and
  train about 158.5k rows (v3 train 160,536 minus the 2000 extra rows moved into the subsets, minus a few more `cross` removals).
- MedQA: 9178 / 1000 / 1273 with MedQA test intact (a few train rows can drop if they match the larger set of MedMCQA subsets;
  `medqa/train/cross` shows it).
- `resolved_revisions` must equal the pins in `HF_REVISIONS`. If it does not, the pin was not applied.

In [23]:
from src.data.medqa_datasets import build_all, load_splits, build_messages
stats = build_all()

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

README.md:   0%|          | 0.00/654 [00:00<?, ?B/s]

{
  "config": {
    "seed": 42,
    "split_sizes": {
      "medmcqa": {
        "hp_dev": 2000,
        "calib": 1500,
        "thresh": 2500
      },
      "medqa": {
        "val": 1000
      }
    },
    "similarity_filter": {
      "medmcqa": {
        "threshold": 0.85,
        "pool_factor": 2
      }
    },
    "short_stem_words": 10,
    "dedupe_scheme": "two-tier-v1",
    "hf_repos": {
      "medmcqa": "openlifescienceai/medmcqa",
      "medqa": "GBaker/MedQA-USMLE-4-options"
    },
    "hf_revisions": {
      "medmcqa": "91c6572c454088bf71b679ad90aa8dffcd0d5868",
      "medqa": "0fb93dd23a7339b6dcd27e241cb9b5eca62d4d18"
    },
    "split_priority": [
      "test",
      "val",
      "hp_dev",
      "calib",
      "thresh",
      "train"
    ],
    "dataset_order": [
      "medqa",
      "medmcqa"
    ]
  },
  "resolved_revisions": {
    "medmcqa": "91c6572c454088bf71b679ad90aa8dffcd0d5868",
    "medqa": "0fb93dd23a7339b6dcd27e241cb9b5eca62d4d18"
  },
  "environment": {
    "p

## Cell: look at the result

Prints the split sizes for both datasets and one example (from `calib`) in the common format together with the exact prompt the model will see.

In [24]:
for name in ("medmcqa", "medqa"):
    s = load_splits(name)
    print(name, {k: len(v) for k, v in s.items()})

example = load_splits("medmcqa")["calib"][0]
print(example)
print()
for m in build_messages(example):
    print(f"[{m['role']}]\n{m['content']}")

medmcqa {'train': 158510, 'hp_dev': 1998, 'calib': 1499, 'thresh': 2497, 'test': 4168}
medqa {'train': 9178, 'val': 1000, 'test': 1273}
{'id': 'medmcqa-99beb3e3-ae3e-4ec0-a837-740519aae03d', 'source': 'medmcqa', 'question': "All of the following are true about 'secondary apnea' in a neonate EXCEPT:", 'options': {'A': 'The baby is not breathing', 'B': 'HR may become < 100/min', 'C': 'It may be present even at bih', 'D': 'Baby may respond to tactile stimulation'}, 'answer': 'D'}

[user]
All of the following are true about 'secondary apnea' in a neonate EXCEPT:

A. The baby is not breathing
B. HR may become < 100/min
C. It may be present even at bih
D. Baby may respond to tactile stimulation

Answer with only the letter (A, B, C, D).


## Tests

The tests live in the repo's `tests/` folder; the test cell below runs the ones for this module.

**`tests/test_data.py`** (needs the real data)

| Test | What it guarantees |
|---|---|
| `test_split_names` | MedMCQA returns exactly `train, hp_dev, calib, thresh, test`; MedQA returns `train, val, test`. |
| `test_medmcqa_splits_have_no_questions_in_common` | `train`, `hp_dev`, `calib`, `thresh` and `test` share no ids and no `leak_key` (checked for every pair). |
| `test_no_eval_question_in_any_train` | No question of any eval split (both datasets) appears in any train set (the required leakage check). |
| `test_eval_sets_have_no_duplicates` | `leak_key`s are unique across all eval splits of both datasets. |
| `test_train_has_no_exact_duplicates` | `exact_key`s are unique across both train sets. |
| `test_subset_sizes_close_to_nominal` | Each carved subset is within 2% of its nominal size (the shortfall is collision removals). |
| `test_subsets_have_no_near_copy_in_train` | A sample of 300 rows from each of `hp_dev`, `calib`, `thresh` has no different train question at or above the filter threshold (the similarity filter works). |
| `test_ids_unique` | Ids are unique across everything. |
| `test_format_all_rows` | Every row: contiguous letters from `A`, answer is an option, correct `source` and id prefix, non-empty question. |
| `test_medqa_test_is_untouched` | MedQA test equals the official test row for row, so MedAbstain's source is intact. |
| `test_deterministic_in_fresh_processes` | Two builds in two separate Python processes (different `PYTHONHASHSEED`, each with an empty similarity cache so the search is recomputed) give identical id files, and both equal the saved splits in `DATA_DIR`. |
| `test_config_guard` | Changing a setting without rebuilding makes `load_splits` raise. |

**`tests/test_subsets.py`** (offline) checks `carve_subsets` (disjoint, sizes, independent of input order, seed-dependent,
prefix property, bad sizes, flagged rows excluded, pool too small), `near_duplicate_ids` (flags reworded copies, ignores identical
text and unrelated questions, cache gives the same answer) and the guard on subsets (a question in two subsets, or in a subset and in train, is kept once).

**`tests/test_dedupe_logic.py`** (offline) checks the key functions and the two-tier pass on tiny hand-made examples.

**`tests/test_prompts.py`** (offline) checks 4 and 6 option prompts, option-order independence and the message structure.

The data tests use the real splits in `DATA_DIR` (built on first use). The two fresh-process rebuilds make the whole run take
several minutes, because each of the two fresh builds repeats the similarity search.

## Cell: run the tests

Runs both test files. All tests must pass before the splits are shared. Takes two to three minutes (it rebuilds the splits in two fresh Python processes).

In [25]:
!cd {REPO_DIR} && FME_SHARED_DIR={FME_SHARED_DIR} python -m pytest -q tests/test_data.py tests/test_subsets.py tests/test_dedupe_logic.py tests/test_prompts.py

.................................                                        [100%]
33 passed in 486.97s (0:08:06)


## Cell: reference sweep, how many MedMCQA train rows each key removes

This cell only explains the design; the pipeline does not use it. On the reference run it printed 30652 (question only, v1), 27165,
23672, 21542 (threshold 10, v2), 20168, 19226 and 18106 (question + options, now `exact_key` for train). Rerun it any time
to see the same numbers.

It compares how many MedMCQA train rows would be removed under different keys:

- threshold `0`: question text only (the v1 rule, expected to reproduce 30652);
- thresholds `6` to `15`: question-only for stems at least that many words long, question + options for shorter stems;
- threshold `1000000`: question + options for everything (exact duplicates only).

The second output shows where the question-only removals come from, by stem length. Reading it:

- The drop from 30652 to 18106 is distinct questions that share a stem but have different options; v1 wrongly discarded them.
- The remaining 18106 are true copies, about 10% of MedMCQA train.
- The gap between 21542 (threshold 10) and 18106 is long stems with different options. In train these are now kept (two-tier);
  against val/test they are still treated as leaks (the strict `leak_key`).
- The most repeated short stems are printed so you can eyeball them.

In [26]:
from collections import Counter, defaultdict
from src.data.medqa_datasets.data import load_medmcqa_raw, normalize

rows = load_medmcqa_raw()["train"]

def n_removed(threshold):
    seen, n = set(), 0
    for e in rows:
        q = normalize(e["question"])
        key = q if len(q.split()) >= threshold else q + " || " + " | ".join(sorted(normalize(v) for v in e["options"].values()))
        if key in seen:
            n += 1
        else:
            seen.add(key)
    return n

print("threshold -> rows removed (0 = question-only, 1000000 = question+options)")
for t in (0, 6, 8, 10, 12, 15, 1_000_000):
    print(f"{t:>8} -> {n_removed(t)}")

groups = defaultdict(list)
for e in rows:
    groups[normalize(e["question"])].append(e)
by_len = Counter()
for q, g in groups.items():
    w = len(q.split())
    bucket = "<5 words" if w < 5 else "5-9 words" if w < 10 else "10-19 words" if w < 20 else "20+ words"
    by_len[bucket] += len(g) - 1
print("\nrows removed by question-only rule, by stem length:", dict(by_len))

print("\nmost repeated stems with fewer than 10 words:")
short = sorted(((len(g), q) for q, g in groups.items() if len(q.split()) < 10), reverse=True)[:8]
for count, q in short:
    n_opt_sets = len({tuple(sorted(normalize(v) for v in e["options"].values())) for e in groups[q]})
    print(f"  x{count} ({n_opt_sets} distinct option sets): {q}")

threshold -> rows removed (0 = question-only, 1000000 = question+options)
       0 -> 30652
       6 -> 27165
       8 -> 23672
      10 -> 21542
      12 -> 20168
      15 -> 19226
 1000000 -> 18106

rows removed by question-only rule, by stem length: {'10-19 words': 8000, '5-9 words': 17312, '20+ words': 1826, '<5 words': 3514}

most repeated stems with fewer than 10 words:
  x13 (11 distinct option sets): call exner bodies are seen in
  x8 (5 distinct option sets): schiller duval bodies are seen in
  x8 (6 distinct option sets): hair on end appearance is seen in
  x8 (8 distinct option sets): all of the following statements are true except
  x8 (7 distinct option sets): all are true except
  x7 (2 distinct option sets): whip lash injury is caused due to
  x7 (6 distinct option sets): which of the following is not a prodrug
  x7 (4 distinct option sets): the term id was coined by


## Cell: how different is `test` from train? (near-duplicate check, run before tuning)

MedMCQA's official split is by exam, and its authors built the dev/test sets to differ from train (questions with a Levenshtein
similarity above 0.9 to others were excluded). Our `test` set is the official validation set, so it has this property. The three
new subsets are random slices of train, which still contain many near-copies of other train questions.

If so, a model would look more accurate and more confident on `hp_dev`, `calib` and `thresh` than on `test`, and the temperature and
abstention threshold fitted there would transfer worse than expected. This cell measures it:

- it takes 1000 random train rows and 1000 `test` rows (fixed seed 0);
- for each, it finds the most similar question in train (Levenshtein similarity, 0 to 1; values below 0.5 are reported as 0);
- neighbours whose normalized text is **identical** are ignored, because the leak guard already removes those from train.

Reading the first line (share with a neighbour at 0.9 or more), as a rule of thumb:

- `test` should be close to 0% by construction.
- If random train rows are under about 2%, the shift is negligible: keep the subsets as they are.
- If they are above about 10%, the subsets are easier than `test`: add a similarity filter before freezing (drop candidate subset
  rows that have a train neighbour at 0.9 or more).
- In between, send the output and we decide together.

It uses `rapidfuzz` and takes a few minutes on Colab. It does not change any splits.

**Reference run (before the v5 filter):** 21.8% of random train rows vs 0.5% of `test` rows had a neighbour at 0.9 or more, so
the filter was added. Because this cell looks at the deduplicated train set before carving, it still shows the unfiltered baseline in v5.

In [27]:
!pip -q install rapidfuzz==3.14.6
import random
from collections import defaultdict

import numpy as np
from rapidfuzz import process
from rapidfuzz.distance import Levenshtein
from src.data.medqa_datasets.data import dedupe_within, exact_key, load_medmcqa_raw, normalize

raw = load_medmcqa_raw()
train_rows, _, _ = dedupe_within(raw["train"], exact_key)   # train as the pipeline sees it before carving
train_q = [normalize(e["question"]) for e in train_rows]
test_q = [normalize(e["question"]) for e in raw["test"]]    # official validation = our test

same_text = defaultdict(list)                               # normalized text -> positions in train_q
for i, q in enumerate(train_q):
    same_text[q].append(i)


def nearest_similarity(queries, choices, chunk=100):
    """Highest Levenshtein similarity (0 to 1) of each query to any choice with different text (below 0.5 reported as 0)."""
    best = np.zeros(len(queries), dtype=np.float32)
    for start in range(0, len(queries), chunk):
        block = queries[start:start + chunk]
        sims = process.cdist(block, choices, scorer=Levenshtein.normalized_similarity,
                             score_cutoff=0.5, dtype=np.float32, workers=-1)
        for r, q in enumerate(block):
            sims[r, same_text.get(q, [])] = 0.0             # ignore identical text (includes the row itself)
        best[start:start + chunk] = sims.max(axis=1)
    return best


rng = random.Random(0)
n = 1000
train_sample = rng.sample(train_q, n)
test_sample = rng.sample(test_q, min(n, len(test_q)))
train_best = nearest_similarity(train_sample, train_q)
test_best = nearest_similarity(test_sample, train_q)

print(f"{'share with a train neighbour at least':<40}{'random train rows':>18}{'test rows':>12}")
for t in (0.9, 0.8, 0.7):
    print(f"{t:<40}{(train_best >= t).mean():>18.1%}{(test_best >= t).mean():>12.1%}")
print(f"{'median nearest similarity (0 = below 0.5)':<40}{np.median(train_best):>18.2f}{np.median(test_best):>12.2f}")

share with a train neighbour at least    random train rows   test rows
0.9                                                  21.8%        0.5%
0.8                                                  34.6%        7.4%
0.7                                                  51.5%       22.6%
median nearest similarity (0 = below 0.5)              0.71        0.56


## Cell: did the similarity filter work? (after the build)

Repeats the measurement on the **final** subsets: 1000 random rows from each of `hp_dev`, `calib`, `thresh` (or all rows if fewer)
and 1000 rows of `test`, each compared with the final train set (identical text ignored). The three subsets should now look like
`test`: 0% at and above the filter threshold (guaranteed by construction, which the sampled test also checks) and, as the real
indicator that the shift is reduced, similar shares at 0.8 and 0.7. A small remaining gap below the threshold is expected.

In [28]:
from src.data.medqa_datasets import load_splits

final = load_splits("medmcqa")
final_train = [normalize(e["question"]) for e in final["train"]]
same_text = defaultdict(list)
for i, q in enumerate(final_train):
    same_text[q].append(i)

rows_out = {}
for sp in ("hp_dev", "calib", "thresh", "test"):
    qs = [normalize(e["question"]) for e in final[sp]]
    qs = random.Random(0).sample(qs, min(1000, len(qs)))
    rows_out[sp] = nearest_similarity(qs, final_train)

print(f"{'share with a train neighbour at least':<40}" + "".join(f"{sp:>10}" for sp in rows_out))
for t in (0.9, 0.85, 0.8, 0.7):
    print(f"{t:<40}" + "".join(f"{(v >= t).mean():>10.1%}" for v in rows_out.values()))
print(f"{'median nearest similarity':<40}" + "".join(f"{np.median(v):>10.2f}" for v in rows_out.values()))

share with a train neighbour at least       hp_dev     calib    thresh      test
0.9                                           0.0%      0.0%      0.0%      0.6%
0.85                                          0.3%      0.0%      0.2%      2.7%
0.8                                          11.1%      8.2%     10.0%      8.0%
0.7                                          33.9%     31.5%     31.9%     22.5%
median nearest similarity                     0.64      0.63      0.63      0.56


## Cell: numbers for the README

Prints a Markdown table of the final size of every split of both datasets (with the raw sizes), the removed-duplicate counts,
the number of removed copies whose answer disagreed with the kept copy, and the dataset commits used. Paste the output into the
project README (the "Done when" item).

In [29]:
import json

from src.utils.paths import SPLITS_DIR

s = json.loads((SPLITS_DIR / "stats.json").read_text())
print("| dataset | split | rows |")
print("|---|---|---|")
for n, d in s["sizes"].items():
    for split, rows in d.items():
        print(f"| {n} | {split} | {rows} |")
print("\nRaw sizes (before any removal):", s["raw_sizes"])
print("\nRemoved duplicates (within = copies inside one list; cross = matched an eval question; cross_dup = true copy across the two train sets):")
for k, v in s["removed"].items():
    print(f"- {k}: {v}")
print("\nRemoved duplicates whose answer text disagreed with the kept copy:", s["duplicate_answer_conflicts"])
print("Dataset commits used:", s["resolved_revisions"])
print("Seed:", s["config"]["seed"], "| split sizes:", s["config"]["split_sizes"], "| short-stem threshold:", s["config"]["short_stem_words"])

| dataset | split | rows |
|---|---|---|
| medmcqa | train | 158510 |
| medmcqa | hp_dev | 1998 |
| medmcqa | calib | 1499 |
| medmcqa | thresh | 2497 |
| medmcqa | test | 4168 |
| medqa | train | 9178 |
| medqa | val | 1000 |
| medqa | test | 1273 |

Raw sizes (before any removal): {'medmcqa': {'train': 182822, 'test': 4183}, 'medqa': {'train': 10178, 'test': 1273}}

Removed duplicates (within = copies inside one list; cross = matched an eval question; cross_dup = true copy across the two train sets):
- medmcqa/train/within: 18106
- medmcqa/test/within: 15
- medqa/train/within: 0
- medqa/test/within: 0
- medmcqa/train/cross: 206
- medmcqa/hp_dev/cross: 2
- medmcqa/calib/cross: 1
- medmcqa/thresh/cross: 3
- medmcqa/test/cross: 0
- medqa/train/cross: 0
- medqa/val/cross: 0
- medqa/test/cross: 0
- medmcqa/train/cross_dup: 0
- medqa/train/cross_dup: 0

Removed duplicates whose answer text disagreed with the kept copy: {'medmcqa/train': 400, 'medmcqa/test': 2, 'medqa/train': 0, 'medqa/test

## Cell: checksums of the id files

Prints a SHA-256 checksum of each dataset's id file. Equal checksums mean equal splits, so record them in the README and compare
them after any rebuild or on another machine. The MedQA line is also compared with the v3 reference value: MedQA's carving did
not change in v4, so it should be unchanged unless some MedQA train rows matched the larger set of MedMCQA subsets (see
`medqa/train/cross` above; if that count is 0, the file must be identical).

In [30]:
import hashlib, json

from src.utils.paths import SPLITS_DIR

V3_MEDQA = "6e755d5bbd8c5c256a86e701a0557e1cb8d81e984f6abf7f6d6f57c550ffbc2c"
digests = {}
for p in sorted(SPLITS_DIR.glob("*.json")):
    if p.name != "stats.json":
        digests[p.name] = hashlib.sha256(p.read_bytes()).hexdigest()
        print(p.name, digests[p.name])
print("medqa.json identical to v3:", digests["medqa.json"] == V3_MEDQA)
print("environment of this build:", json.loads((SPLITS_DIR / "stats.json").read_text())["environment"])

medmcqa.json 7bc0abd2202114ea25c13b3d11aafde2fc9c7469db261042eaa4e5c1764ebaad
medqa.json 6e755d5bbd8c5c256a86e701a0557e1cb8d81e984f6abf7f6d6f57c550ffbc2c
medqa.json identical to v3: True
environment of this build: {'python': '3.13.18', 'datasets': '4.8.5', 'rapidfuzz': '3.14.6'}


## Before sharing

1. The dataset commits are already pinned in `HF_REVISIONS`; confirm `resolved_revisions` in `stats.json` equals them.
2. Share the `DATA_DIR` folder (it holds `splits/`, `processed/` and `hf_cache/`) and add it to `.gitignore`; commit only the code.
3. Everyone sets `FME_SHARED_DIR` so that `$FME_SHARED_DIR/data` is that folder, before importing `src.data.medqa_datasets`.
4. Freeze before any tuning: record the checksums from the checksum cell in the README and do not rebuild afterwards. A rebuild can change ids, which would invalidate runs that already used the old subsets.